### 1. Fetch credentials securely from Databricks Secrets

In [ ]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

SCOPE_NAME = "hospital-scope"
STORAGE_ACCOUNT_KEY = dbutils.secrets.get(scope=SCOPE_NAME, key="storage-account-key")

### 2. Configuration Variables

In [ ]:
STORAGE_ACCOUNT_NAME = "mhahospitalstorage"
BRONZE_CONTAINER = "bronze"
SILVER_CONTAINER = "silver"

### 3. Safe configuration for Spark Session

In [ ]:
spark.conf.set(f"fs.azure.account.key.{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net", STORAGE_ACCOUNT_KEY)

bronze_path = f"abfss://{BRONZE_CONTAINER}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/patient_flow_raw"
silver_path = f"abfss://{SILVER_CONTAINER}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/patient_flow_cleaned"

### 4. Read from bronze Delta table

In [ ]:
bronze_df = (
    spark.readStream
    .format("delta")
    .load(bronze_path)
)

### 5. Define Schema

In [ ]:
schema = StructType([
    StructField("patient_id", StringType()),
    StructField("gender", StringType()),
    StructField("age", IntegerType()),
    StructField("department", StringType()),
    StructField("admission_time", StringType()),
    StructField("discharge_time", StringType()),
    StructField("bed_id", IntegerType()),
    StructField("hospital_id", IntegerType())
])

### 6. Parse JSON and transform

In [ ]:
parsed_df = bronze_df.select(from_json(col("raw_json"), schema).alias("data")).select("data.*")

### 7. Data Cleaning Transformations

In [ ]:
clean_df = (parsed_df
    .withColumn("admission_time", to_timestamp("admission_time"))
    .withColumn("discharge_time", to_timestamp("discharge_time"))
    .withColumn("admission_time", 
                when(col("admission_time").isNull() | (col("admission_time") > current_timestamp()), current_timestamp())
                .otherwise(col("admission_time")))
    .withColumn("age", 
                when(col("age") > 100, floor(rand()*90 + 1).cast("int"))
                .otherwise(col("age")))
)

### 8. Schema enforcement/evolution

In [ ]:
for col_name in schema.names:
    if col_name not in clean_df.columns:
        clean_df = clean_df.withColumn(col_name, lit(None))

### 9. Write to silver table

In [ ]:
(
    clean_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{silver_path}_checkpoint")
    .option("mergeSchema", "true")
    .start(silver_path)
)